# 地方競馬 予想システム 〜 適性の特徴量の採用検討（3年 × 乱数10回・保存はしない）

前回、③適性（19個）は「市場との差」で25勝5敗と精度が上がったが、**今の買い基準（期待値1.2・度合い1.6）のままでは回収率が下がった**。
買い基準は今のモデルに合わせて決めたものなので、**モデルごとに買い基準を広く振って**比べ直す。

| 案 | 中身 |
|---|---|
| 今のモデル | 基準 |
| ③全部 | 適性の特徴量19個をすべて追加 |
| ③上位 | 前回よく使われた6個だけ追加（前走より相手が強いか・直近3走の相手のレベル・コース適性の指数・騎手×調教師・前走と5走平均の指数の差・道悪の指数） |

- **単勝**: 期待値 1.0〜1.5 × 1着率の下限 5〜20%（1レース1頭）
- **ワイド**: 度合い 1.2〜2.0 × ワイド確率の下限 0.10〜0.30（1レース1点）
- 各マスで年ごとの回収率・3年合計・的中率を出し、**「③が今のモデルを上回ったマスの数」**と、**各モデルの「3年とも100%を超えるマス」**を一覧にする
  （1つの良いマスだけを選ぶと偶然を拾うので、マス全体の傾向で判断する）

**Input**: 学習用と同じ3つ。Internet は不要。**実行**: Save & Run All (Commit)。**3〜4時間**の見込み。

## ⓪ 設定

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

DATA_ROOT   = "/kaggle/input"
TEST_YEARS  = list(range(2016, 2026))  # 1年ずつ評価する年（各年とも「前年まで」のデータだけで学習）
BASE_END    = "2015-01-01"   # スピード指数の基準タイムはこの日より前のデータだけで作る（全ての評価年より過去）
USE_ODDS    = True          # True: オッズも特徴量に入れ、「市場の評価のどこがズレているか」を直接学習させる

## ① 読込（オッズ "2,012.0" のカンマで割れた行を直して読む）

In [ ]:
DIR = os.path.dirname(glob.glob(f"{DATA_ROOT}/**/keiba_results.csv", recursive=True)[0])

def fix_row(row):
    if len(row) == 19:
        return row[:15] + [row[15] + row[16]] + row[17:]
    return None

res = pd.read_csv(f"{DIR}/keiba_results.csv", engine="python", on_bad_lines=fix_row,
                  dtype={"race_id": str, "horse_id": str, "jockey_id": str})
races = pd.read_csv(f"{DIR}/keiba_races.csv", dtype={"race_id": str})
pays  = pd.read_csv(f"{DIR}/keiba_payouts.csv", dtype={"race_id": str, "horse_num": str})
# 追加データ（2026年収集分・過去の穴埋め分）。収集ノートブックの出力を Private データセットにして Add Input しておくと自動で結合
for tag in ["2026", "fill"]:
    add = {n: sorted(glob.glob(f"{DATA_ROOT}/**/keiba_{n}_{tag}.csv", recursive=True)) for n in ["results", "races", "payouts"]}
    if not all(add.values()):
        continue
    res = pd.concat([res] + [pd.read_csv(p, dtype={"race_id": str, "horse_id": str, "jockey_id": str}) for p in add["results"]],
                    ignore_index=True).drop_duplicates(["race_id", "number"], keep="last")
    races = pd.concat([races] + [pd.read_csv(p, dtype={"race_id": str}) for p in add["races"]],
                      ignore_index=True).drop_duplicates("race_id", keep="last")
    pays = pd.concat([pays] + [pd.read_csv(p, dtype={"race_id": str, "horse_num": str}) for p in add["payouts"]],
                     ignore_index=True).drop_duplicates(keep="last")
    print(f"追加データ（{tag}）を結合しました:", add["races"])
print("results:", res.shape, "/ races:", races.shape, "/ payouts:", pays.shape)

## ② 地方に絞り、race_id から日付を作る

地方の race_id は「年(4) + 場(2) + 月(2) + 日(2) + R(2)」。帯広（ばんえい, 場コード65）は別競技なので除外。
下の確認で「月」が 1〜12 だけ、「日」が 1〜31 だけになっていれば解釈は正しい。

In [ ]:
EXCLUDE_VENUES = {"65"}

def is_nar(rid):
    code = rid.str[4:6]
    return ~code.astype(int).between(1, 10) & ~code.isin(EXCLUDE_VENUES)

res   = res[is_nar(res["race_id"])].copy()
races = races[is_nar(races["race_id"])].copy()

mm = races["race_id"].str[6:8].astype(int)
dd = races["race_id"].str[8:10].astype(int)
print("月の範囲:", mm.min(), "〜", mm.max(), "/ 日の範囲:", dd.min(), "〜", dd.max())
print("月ごとのレース数:", mm.value_counts().sort_index().to_dict())
assert mm.between(1, 12).all() and dd.between(1, 31).all(), "月日として解釈できない値があります"

races["date"] = pd.to_datetime(races["race_id"].str[:4] + races["race_id"].str[6:10],
                               format="%Y%m%d", errors="coerce")
print("日付が作れなかったレース:", races["date"].isna().sum())
races = races.dropna(subset=["date"])
pays  = pays[pays["race_id"].isin(set(races["race_id"]))].copy()
print("地方レース数:", len(races), "/ 期間:", races["date"].min().date(), "〜", races["date"].max().date())

## ③ 整形

In [ ]:
race_cols = ["race_id", "date", "venue", "race_number", "course_type", "distance",
             "turn", "weather", "track_condition"]
df = res.merge(races[race_cols], on="race_id", how="inner")

# 着順: 取消・除外は出走していないので除く / 中止・失格は最下位扱い / "2(降)" は数字部分
df = df[~df["rank"].astype(str).isin(["取", "除"])].copy()
df["n_runners"] = df.groupby("race_id")["race_id"].transform("size")
df["finish"] = pd.to_numeric(df["rank"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce")
df["finish"] = df["finish"].fillna(df["n_runners"]).astype(int)
df["win"]  = (df["finish"] == 1).astype(int)
df["top3"] = (df["finish"] <= 3).astype(int)
df["finish_ratio"] = (df["finish"] - 1) / (df["n_runners"] - 1).clip(lower=1)

# 数値化
for c in ["frame", "number", "weight", "distance", "race_number", "odds", "popularity", "last_3f"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["sex"] = df["sex_age"].astype(str).str[0]
df["age"] = pd.to_numeric(df["sex_age"].astype(str).str[1:], errors="coerce")
bw = df["horse_weight"].astype(str).str.extract(r"(\d+)\(([-+]?\d+)\)")
df["body_weight"] = pd.to_numeric(bw[0], errors="coerce")
df["bw_diff"]     = pd.to_numeric(bw[1], errors="coerce")
t = df["time"].astype(str).str.extract(r"(?:(\d+):)?(\d+\.\d)")
df["time_sec"] = pd.to_numeric(t[0], errors="coerce").fillna(0) * 60 + pd.to_numeric(t[1], errors="coerce")
pos = df["passing"].astype(str).str.split("-")
df["first_corner"] = pd.to_numeric(pos.str[0], errors="coerce")
df["horse_key"]  = df["horse_id"].fillna(df["horse_name"])
df["jockey_key"] = df["jockey_id"].fillna(df["jockey_name"])
df["month"] = df["date"].dt.month

# レース後にしか分からない「そのレースでの内容」（前走特徴量の材料。そのレースの特徴量には使わない）
df["time_behind"]  = df["time_sec"] - df.groupby("race_id")["time_sec"].transform("min")
df["last3f_rank"]  = df.groupby("race_id")["last_3f"].rank(pct=True)
df["corner1_ratio"] = df["first_corner"] / df["n_runners"]
print(df.shape)

## ④ 過去走の特徴量（必ず shift(1) で当該レースを除外）

スピード指数は「直近3年の基準タイム」と「その日の馬場差」で計算（2026/10 改良）。

F4（メンバーの中での位置: 5走平均・最高指数のメンバー内の偏差、最上位との差、メンバー上位3頭の平均との差）も追加（2026/11）。

In [ ]:
# --- スピード指数: 競馬場・コース・距離・馬場ごとの基準タイムとの差（50が平均、高いほど速い）---
# 基準タイムは BASE_END より前の5着以内の馬だけで作る（どの評価年から見ても過去のデータ）
BASE_KEYS = ["venue", "course_type", "distance", "track_condition"]
base = (df[(df["date"] < BASE_END) & (df["finish"] <= 5)]
        .groupby(BASE_KEYS)["time_sec"].agg(base_mean="mean", base_std="std", base_n="size").reset_index())
base = base[base["base_n"] >= 30]
df = df.drop(columns=["base_mean", "base_std", "base_n", "speed"], errors="ignore").merge(base, on=BASE_KEYS, how="left")
df["speed"] = (df["base_mean"] - df["time_sec"]) / df["base_std"] * 10 + 50
print("スピード指数が計算できた割合:", round(df["speed"].notna().mean(), 3))

# --- スピード指数の作り直し（2026/10 改良：比較実験で5回全勝）---
# 1) 基準タイムを「各年について、その前の3年」で作り直す（馬場の改修や新しい距離設定に追随。未来の年は使わない）
# 2) その日・その競馬場・コースの時計の速さ（馬場差）で補正する（レース後に分かる値で、次走以降の特徴量にだけ使う）
df["year"] = df["date"].dt.year
df["_bk"] = (df["venue"].astype(str) + "|" + df["course_type"].astype(str) + "|" + df["distance"].astype(str)
             + "|" + df["track_condition"].astype(str))
parts = []
for Y in sorted(df["year"].unique()):
    s_ = df[(df["year"] >= Y - 3) & (df["year"] <= Y - 1) & (df["finish"] <= 5)]
    st = s_.groupby("_bk")["time_sec"].agg(b2_mean="mean", b2_std="std", b2_n="size").reset_index()
    st = st[st["b2_n"] >= 30]
    st["year"] = Y
    parts.append(st)
df = df.drop(columns=["b2_mean", "b2_std", "b2_n"], errors="ignore").merge(pd.concat(parts, ignore_index=True), on=["_bk", "year"], how="left")
raw2 = ((df["b2_mean"] - df["time_sec"]) / df["b2_std"] * 10 + 50).fillna(df["speed"])
race_dev = raw2.where(df["finish"] <= 5).groupby(df["race_id"]).transform("mean") - 50
day = df["date"].astype(str) + "_" + df["venue"].astype(str) + "_" + df["course_type"].astype(str)
first = ~df["race_id"].duplicated()
day_dev = race_dev[first].groupby(day[first]).mean()
n_day = first.groupby(day).sum()
variant = day.map(day_dev).where(day.map(n_day) >= 3)
df["speed_old"] = df["speed"]
df["speed"] = raw2 - variant.fillna(0)
print("新しいスピード指数: 直近3年の基準が使えた割合", round(df["b2_mean"].notna().mean(), 3),
      "/ 馬場差で補正できた割合", round(variant.notna().mean(), 3), "/ 計算できた割合", round(df["speed"].notna().mean(), 3))

# 予想用ノートブックで結果を手入力したときに使う基準タイム（直近3年。馬場差の補正は無し）
LY = int(df["year"].max())
base = (df[(df["year"] >= LY - 2) & (df["finish"] <= 5)]
        .groupby(BASE_KEYS, observed=True)["time_sec"].agg(base_mean="mean", base_std="std", base_n="size").reset_index())
base = base[base["base_n"] >= 30]

df = df.sort_values(["date", "race_id", "number"]).reset_index(drop=True)
g = df.groupby("horse_key", sort=False)

def past_mean(col, n):
    s = g[col].shift(1)
    return s.groupby(df["horse_key"]).rolling(n, min_periods=1).mean().reset_index(level=0, drop=True)

for k in (1, 2, 3):
    df[f"prev_finish_{k}"] = g["finish"].shift(k)
df["prev_finish_ratio"]   = g["finish_ratio"].shift(1)
df["prev_speed"]          = g["speed"].shift(1)
df["avg_speed_5"]         = past_mean("speed", 5)
df["max_speed_5"]         = (g["speed"].shift(1).groupby(df["horse_key"]).rolling(5, min_periods=1).max()
                             .reset_index(level=0, drop=True))
df["avg_finish_ratio_5"]  = past_mean("finish_ratio", 5)
df["top3_rate_5"]         = past_mean("top3", 5)
df["win_rate_10"]         = past_mean("win", 10)
df["prev_time_behind"]    = g["time_behind"].shift(1)
df["avg_time_behind_5"]   = past_mean("time_behind", 5)
df["prev_last3f_rank"]    = g["last3f_rank"].shift(1)
df["avg_last3f_rank_5"]   = past_mean("last3f_rank", 5)
df["prev_corner1_ratio"]  = g["corner1_ratio"].shift(1)
df["avg_corner1_ratio_5"] = past_mean("corner1_ratio", 5)
df["prev_popularity"]     = g["popularity"].shift(1)
df["prev_log_odds"]       = np.log1p(g["odds"].shift(1))
df["n_races"]             = g.cumcount()
df["days_since_last"]     = (df["date"] - g["date"].shift(1)).dt.days
df["dist_diff"]           = df["distance"] - g["distance"].shift(1)
df["weight_diff"]         = df["weight"] - g["weight"].shift(1)
df["same_venue_as_prev"]  = (df["venue"] == g["venue"].shift(1)).astype(float)

# 騎手・調教師の通算成績（前日までの成績のみ。平滑化つき）
K = 30
def past_rate(key, target):
    daily = (df.groupby([key, "date"])[target].agg(["sum", "count"])
               .reset_index().sort_values([key, "date"]))
    gg = daily.groupby(key)
    past_sum = gg["sum"].cumsum() - daily["sum"]
    past_cnt = gg["count"].cumsum() - daily["count"]
    p = df[target].mean()
    daily[f"{key}_{target}_rate"] = (past_sum + p * K) / (past_cnt + K)
    return daily[[key, "date", f"{key}_{target}_rate"]]

for key, target in [("jockey_key", "top3"), ("jockey_key", "win"), ("trainer", "top3")]:
    df = df.merge(past_rate(key, target), on=[key, "date"], how="left")

# レース内での相対位置（メンバーの中で強いか）
for c in ["avg_finish_ratio_5", "top3_rate_5", "avg_time_behind_5", "jockey_key_top3_rate",
          "avg_speed_5", "max_speed_5"]:
    df[f"{c}_inrace"] = df.groupby("race_id")[c].rank(pct=True)

# F4 メンバーの中での位置（2026/11 改良：3年×乱数10回の検証で26勝4敗）
for c in ["avg_speed_5", "max_speed_5"]:
    gc = df.groupby("race_id")[c]
    df[f"{c}_z"] = (df[c] - gc.transform("mean")) / gc.transform("std").replace(0, np.nan)
df["max_speed_5_gap"] = df["max_speed_5"] - df.groupby("race_id")["max_speed_5"].transform("max")
rk = df.groupby("race_id")["max_speed_5"].rank(ascending=False, method="first")
df["field_level"] = df["max_speed_5"].where(rk <= 3).groupby(df["race_id"]).transform("mean")
df["max_speed_5_vs_field"] = df["max_speed_5"] - df["field_level"]
F4_COLS = ["avg_speed_5_z", "max_speed_5_z", "max_speed_5_gap", "field_level", "max_speed_5_vs_field"]

cat_cols = ["venue", "course_type", "turn", "weather", "track_condition", "sex"]
for c in cat_cols:
    df[c] = df[c].astype("category")

features = (
    ["frame", "number", "weight", "age", "body_weight", "bw_diff", "distance", "race_number",
     "n_runners", "month"] + cat_cols +
    ["prev_finish_1", "prev_finish_2", "prev_finish_3", "prev_finish_ratio", "avg_finish_ratio_5",
     "prev_speed", "avg_speed_5", "max_speed_5",
     "top3_rate_5", "win_rate_10", "prev_time_behind", "avg_time_behind_5", "prev_last3f_rank",
     "avg_last3f_rank_5", "prev_corner1_ratio", "avg_corner1_ratio_5", "prev_popularity",
     "prev_log_odds", "n_races", "days_since_last", "dist_diff", "weight_diff", "same_venue_as_prev",
     "jockey_key_top3_rate", "jockey_key_win_rate", "trainer_top3_rate"] +
    [c for c in df.columns if c.endswith("_inrace")] + F4_COLS
)
if USE_ODDS:
    features.append("odds")
print(len(features), "features")

## ④-2 ③ 適性の特徴量（すべて「そのレースより前」の成績だけで作る）

In [ ]:
import time
t0 = time.time()
df = df.sort_values(["date", "race_id", "number"]).reset_index(drop=True)
df["dist_band"] = pd.cut(df["distance"], [0, 1200, 1600, 2000, 9999], labels=["短距離", "マイル", "中距離", "長距離"]).astype(str)
df["going"] = np.where(df["track_condition"].astype(str).isin(["重", "不良"]), "道悪", "良系")
df["_t3"] = df["top3"].fillna(0).astype(float)
df["_sp"] = df["speed"].fillna(0).astype(float)
df["_spok"] = df["speed"].notna().astype(float)
PRI3, KS = df["_t3"].mean(), 5

def horse_cond(keys, name):
    # 馬ごと・条件ごとの過去の3着内率（平滑化）と、平均スピード指数の「その馬の全体の平均」との差
    g = df.groupby(["horse_key"] + keys, observed=True, sort=False)
    n = g.cumcount()
    s3 = g["_t3"].cumsum() - df["_t3"]
    ssp = g["_sp"].cumsum() - df["_sp"]
    nsp = g["_spok"].cumsum() - df["_spok"]
    df[f"{name}_top3"] = (s3 + PRI3 * KS) / (n + KS)
    df[f"{name}_n"] = n
    df[f"{name}_speed"] = ssp / nsp.replace(0, np.nan)

horse_cond([], "h_all")
horse_cond(["venue", "course_type", "dist_band"], "apt_course")
horse_cond(["dist_band"], "apt_dist")
horse_cond(["going"], "apt_going")
for nm in ["apt_course", "apt_dist", "apt_going"]:
    df[f"{nm}_speed_diff"] = df[f"{nm}_speed"] - df["h_all_speed"]
    df[f"{nm}_top3_diff"] = df[f"{nm}_top3"] - df["h_all_top3"]

def past_rate_multi(keys, name, K_=30):
    daily = (df.groupby(keys + ["date"], observed=True)["_t3"].agg(["sum", "count"]).reset_index().sort_values(keys + ["date"]))
    gg = daily.groupby(keys, observed=True)
    ps, pc = gg["sum"].cumsum() - daily["sum"], gg["count"].cumsum() - daily["count"]
    daily[name] = (ps + PRI3 * K_) / (pc + K_)
    daily[name + "_n"] = pc
    return daily[keys + ["date", name, name + "_n"]]

for keys, name in [(["jockey_key", "trainer"], "jt_top3_rate"), (["jockey_key", "venue"], "jv_top3_rate")]:
    df = df.merge(past_rate_multi(keys, name), on=keys + ["date"], how="left")
df = df.sort_values(["date", "race_id", "number"]).reset_index(drop=True)

gh = df.groupby("horse_key", sort=False)
df["prev_field_level"] = gh["field_level"].shift(1)
df["avg_field_level_3"] = gh["field_level"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df["prev_speed_vs_avg5"] = df["prev_speed"] - df["avg_speed_5"]
df["speed_trend_3"] = gh["speed"].shift(1) - gh["speed"].shift(3)
df["field_vs_prev_field"] = df["field_level"] - df["prev_field_level"]      # 前走より相手が強いか

APT_COLS = ["apt_course_top3", "apt_course_n", "apt_course_speed_diff", "apt_course_top3_diff",
            "apt_dist_top3", "apt_dist_speed_diff", "apt_dist_top3_diff",
            "apt_going_top3", "apt_going_n", "apt_going_speed_diff", "apt_going_top3_diff",
            "jt_top3_rate", "jt_top3_rate_n", "jv_top3_rate",
            "prev_field_level", "avg_field_level_3", "field_vs_prev_field", "prev_speed_vs_avg5", "speed_trend_3"]
print(f"③ 適性の特徴量 {len(APT_COLS)}個 / {time.time() - t0:.0f}秒")
print(df[APT_COLS].describe().T[["count", "mean", "min", "max"]].round(3).to_string())

## ⑤ 比較の部品

In [ ]:
from scipy.optimize import minimize
import re
from itertools import combinations

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
df["year"] = df["date"].dt.year
inv = 1 / df["odds"].where(df["odds"] > 0)
df["p_mkt"] = inv / inv.groupby(df["race_id"]).transform("sum")
df["race_ok"] = (df.groupby("race_id")["p_mkt"].transform(lambda s: s.notna().all())
                 & (df.groupby("race_id")["win"].transform("sum") == 1))
df["rel"] = np.clip(4 - df["finish"].fillna(99), 0, 3).astype(int)     # レース単位の学習用: 1着3・2着2・3着1・他0
LAST = int(df["year"].max())
YEARS = [LAST, LAST - 1, LAST - 2]
order = ["date", "race_id", "number"]
base_params = dict(n_estimators=3000, learning_rate=0.05, num_leaves=63, min_child_samples=100,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)

def race_norm(x, rid):
    x = pd.Series(np.asarray(x, dtype=float), index=rid.index)
    return (x / x.groupby(rid).transform("sum")).values

def race_softmax(score, rid):
    s = pd.Series(np.asarray(score, dtype=float), index=rid.index)
    e = np.exp(s - s.groupby(rid).transform("max"))
    return (e / e.groupby(rid).transform("sum")).values

def comb(d, w):
    s = w[0] * np.log(d["pm"].clip(1e-6)) + w[1] * np.log(d["p_mkt"].clip(1e-6))
    e = np.exp(s - s.groupby(d["race_id"]).transform("max"))
    return e / e.groupby(d["race_id"]).transform("sum")

def ll(w, d):
    return -np.log(comb(d, w)[d["win"] == 1].clip(1e-9)).mean()

def gap(d_fit, d_ev):
    w = minimize(ll, x0=[0.5, 0.5], args=(d_fit,), method="Nelder-Mead").x
    return ll([0, 1], d_ev) - ll(w, d_ev), w

def set_period(Y):
    global tr, va_fit, va_ev
    tr = df[df["year"] <= Y - 1].sort_values(order)
    v_ = df[(df["year"] == Y) & df["race_ok"]].sort_values(order)
    mid_ = v_["date"].drop_duplicates().sort_values().iloc[v_["date"].nunique() // 2]
    va_fit, va_ev = v_[v_["date"] < mid_], v_[v_["date"] >= mid_]
    print(f"\n===== {Y}年: 学習 〜{Y - 1}年 {tr['race_id'].nunique():,}レース / 前半 {va_fit['race_id'].nunique():,} / 後半（採点）{va_ev['race_id'].nunique():,}（{mid_.date()}〜） =====")

def groups(d):
    return d.groupby("race_id", sort=False).size().values

def fit_predict(feats, mode, sd):
    if mode == "rank":
        m = lgb.LGBMRanker(**{**base_params, "random_state": sd, "objective": "lambdarank"})
        m.fit(tr[feats], tr["rel"], group=groups(tr), eval_set=[(va_fit[feats], va_fit["rel"])],
              eval_group=[groups(va_fit)], eval_at=[1], callbacks=[lgb.early_stopping(100, verbose=False)])
        pf = race_softmax(m.predict(va_fit[feats]), va_fit["race_id"])
        pe = race_softmax(m.predict(va_ev[feats]), va_ev["race_id"])
    else:
        m = lgb.LGBMClassifier(**{**base_params, "random_state": sd})
        m.fit(tr[feats], tr["win"], eval_set=[(va_fit[feats], va_fit["win"])], eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        pf = race_norm(m.predict_proba(va_fit[feats])[:, 1], va_fit["race_id"])
        pe = race_norm(m.predict_proba(va_ev[feats])[:, 1], va_ev["race_id"])
    return pf, pe, m.best_iteration_, m

COLS = ["race_id", "date", "number", "win", "p_mkt", "odds", "n_runners"]
def run(name, feats, mode):
    t0 = time.time()
    pfs, pes, its, sg = [], [], [], []
    for sd in SEEDS:
        pf, pe, it, m = fit_predict(feats, mode, sd)
        pfs.append(pf); pes.append(pe); its.append(it)
        sg.append(gap(va_fit[COLS].assign(pm=pf), va_ev[COLS].assign(pm=pe))[0])
    pf, pe = np.mean(pfs, axis=0), np.mean(pes, axis=0)
    g1, w1 = gap(va_fit[COLS].assign(pm=pf), va_ev[COLS].assign(pm=pe))
    E = va_ev[COLS].assign(pm=pe)
    E["p"] = comb(E, w1).values
    E["ev"] = E["p"] * E["odds"]
    r = dict(市場との差=g1, 木=int(np.mean(its)), 秒=int(time.time() - t0), _seed=sg, _w=w1, _E=E,
             _imp=dict(zip(feats, m.feature_importances_)))
    print(f"{name}: 市場との差 {g1:.5f}（乱数ごと " + " / ".join(f"{x:.5f}" for x in sg)
          + f"）/ 合成の重み モデル {w1[0]:.3f}・市場 {w1[1]:.3f} / 木 {r['木']} / {r['秒']}秒")
    return r

# ---------- 回収率（案F の単勝・度合い1.6以上のワイド）----------
wp = pays[pays["bet_type"] == "ワイド"].copy()
def _pk(s):
    nums = sorted(int(x) for x in re.findall(r"\d+", str(s)))
    return f"{nums[0]}-{nums[1]}" if len(nums) == 2 else None
wp["key"] = wp["horse_num"].map(_pk)
wp["payout"] = pd.to_numeric(wp["payout"].astype(str).str.replace(",", ""), errors="coerce")
WPAY = wp.dropna(subset=["key", "payout"]).groupby(["race_id", "key"])["payout"].max().to_dict()
HAVE_WIDE = set(wp["race_id"])
N_SIM = 2000
GUM, MKT_MAT = {}, {}

def pair_mat(p, g, k):
    s = g + np.log(np.clip(p, 1e-9, None))
    top = np.argpartition(-s, k - 1, axis=1)[:, :k]
    hit = np.zeros(s.shape, dtype=np.float32)
    np.put_along_axis(hit, top, 1.0, axis=1)
    return (hit.T @ hit) / len(s)

def roi(E):
    s = E[(E["ev"] >= 1.2) & (E["p"] >= 0.10)]
    s = s.loc[s.groupby("race_id")["ev"].idxmax()] if len(s) else s
    tan = (s["odds"] * s["win"]).values
    wide = []
    for rid, X in E[E["race_id"].isin(HAVE_WIDE) & (E["n_runners"] >= 5)].groupby("race_id", sort=False):
        n = len(X); k = 3 if n >= 8 else 2
        if rid not in GUM:
            GUM[rid] = np.random.default_rng(abs(hash(rid)) % 2**32).gumbel(size=(N_SIM, n))
            MKT_MAT[rid] = pair_mat(X["p_mkt"].values, GUM[rid], k)
        mm = pair_mat(X["p"].values, GUM[rid], k)
        iu = np.triu_indices(n, 1)
        pm, pk = mm[iu], MKT_MAT[rid][iu]
        ok = (pm >= 0.2) & (pm / np.maximum(pk, 1e-9) >= 1.6)
        if ok.any():
            j = np.argmax(np.where(ok, pm / np.maximum(pk, 1e-9), -1))
            nums = X["number"].astype(int).values
            a, b = sorted((nums[iu[0][j]], nums[iu[1][j]]))
            wide.append(WPAY.get((rid, f"{a}-{b}"), 0) / 100)
    return tan, np.array(wide)

In [ ]:
# ---------- 買い基準を広く振るための材料を集める（モデルごと・年ごと）----------
def collect(E, Y):
    T_ = E[["race_id", "date", "p", "ev", "odds", "win"]].assign(年=Y)
    rows = []
    for rid, X in E[E["race_id"].isin(HAVE_WIDE) & (E["n_runners"] >= 5)].groupby("race_id", sort=False):
        n = len(X); k = 3 if n >= 8 else 2
        if rid not in GUM:
            GUM[rid] = np.random.default_rng(abs(hash(rid)) % 2**32).gumbel(size=(N_SIM, n))
            MKT_MAT[rid] = pair_mat(X["p_mkt"].values, GUM[rid], k)
        mm = pair_mat(X["p"].values, GUM[rid], k)
        iu = np.triu_indices(n, 1)
        pm, pk = mm[iu], MKT_MAT[rid][iu]
        keep = pm >= 0.10
        if not keep.any():
            continue
        nums = X["number"].astype(int).values
        for a_i, b_i, m_, k_ in zip(iu[0][keep], iu[1][keep], pm[keep], pk[keep]):
            a, b = sorted((nums[a_i], nums[b_i]))
            rows.append((rid, Y, m_, m_ / max(k_, 1e-9), WPAY.get((rid, f"{a}-{b}"), 0) / 100))
    W_ = pd.DataFrame(rows, columns=["race_id", "年", "pw", "ratio", "ret"])
    return T_, W_

## ⑥ 比較（3年 × 乱数10回）

In [ ]:
APT_TOP = ["field_vs_prev_field", "avg_field_level_3", "apt_course_speed_diff", "jt_top3_rate",
           "prev_speed_vs_avg5", "apt_going_speed_diff"]
CMP = {"今のモデル": (features, "cls"), "③全部": (features + APT_COLS, "cls"), "③上位": (features + APT_TOP, "cls")}
ALL, TAN, WID = {}, {k: [] for k in CMP}, {k: [] for k in CMP}
T_START = time.time()
for Y in YEARS:
    set_period(Y)
    ALL[Y] = {}
    for name, (feats, mode) in CMP.items():
        ALL[Y][name] = run(f"{Y} {name}", feats, mode)
        t_, w_ = collect(ALL[Y][name]["_E"], Y)
        TAN[name].append(t_); WID[name].append(w_)
    print(f"（ここまで {(time.time() - T_START) / 3600:.1f}時間）")
TAN = {k: pd.concat(v, ignore_index=True) for k, v in TAN.items()}
WID = {k: pd.concat(v, ignore_index=True) for k, v in WID.items()}

def diffs(a, b, years=YEARS):
    return np.concatenate([np.array(ALL[Y][a]["_seed"]) - np.array(ALL[Y][b]["_seed"]) for Y in years])
print(f"\n【年ごとの「市場との差」（かっこ内は今のモデルとの差）】")
print(pd.DataFrame([{"年": Y, **{k: f"{ALL[Y][k]['市場との差']:.5f}（{ALL[Y][k]['市場との差'] - ALL[Y]['今のモデル']['市場との差']:+.5f}）" for k in CMP}} for Y in YEARS]).to_string(index=False))
print("\n【勝ち負け】（30回中24回以上の勝ちで効果あり）")
for a, b in [("③全部", "今のモデル"), ("③上位", "今のモデル"), ("③上位", "③全部")]:
    d = diffs(a, b); w_ = int((d > 0).sum())
    v_ = "効果あり" if w_ >= 24 and d.mean() > 0 else ("逆効果" if w_ <= 6 else "誤差の範囲")
    print(f"▼ {a} と {b}: {w_}勝{30 - w_}敗（{v_}）/ 平均の差 {d.mean():+.5f}（" +
          " / ".join(f"{Y}年 {int((diffs(a, b, [Y]) > 0).sum())}勝" for Y in YEARS) + "）")

## ⑦ 買い基準を広く振った回収率

In [ ]:
EV_GRID, PT_GRID = [1.0, 1.1, 1.2, 1.3, 1.4, 1.5], [0.05, 0.10, 0.15, 0.20]
R_GRID, PW_GRID = [1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 2.0], [0.10, 0.15, 0.20, 0.25, 0.30]

def tan_pick(T_, ev_min, p_min):
    s = T_[(T_["ev"] >= ev_min) & (T_["p"] >= p_min)]
    return s.loc[s.groupby("race_id")["ev"].idxmax()] if len(s) else s
def wide_pick(W_, r_min, pw_min):
    s = W_[(W_["ratio"] >= r_min) & (W_["pw"] >= pw_min)]
    return s.loc[s.groupby("race_id")["ratio"].idxmax()] if len(s) else s

def grid(kind):
    out = {}
    for name in CMP:
        for a in (EV_GRID if kind == "tan" else R_GRID):
            for b in (PT_GRID if kind == "tan" else PW_GRID):
                s = tan_pick(TAN[name], a, b) if kind == "tan" else wide_pick(WID[name], a, b)
                ret = s["odds"] * s["win"] if kind == "tan" else s["ret"]
                yr = [ret[s["年"] == Y].mean() if (s["年"] == Y).any() else np.nan for Y in YEARS]
                out[(name, a, b)] = dict(n=len(s), roi=ret.mean() if len(s) else np.nan, hit=(ret > 0).mean() if len(s) else np.nan,
                                         yr=yr, ny=[int((s["年"] == Y).sum()) for Y in YEARS])
    return out

for kind, A_, B_, an, bn in [("tan", EV_GRID, PT_GRID, "期待値", "1着率"), ("wide", R_GRID, PW_GRID, "度合い", "ワイド確率")]:
    G = grid(kind)
    title = "単勝（1レース1頭・期待値最大）" if kind == "tan" else "ワイド（1レース1点・度合い最大）"
    print(f"\n{'=' * 20} {title} {'=' * 20}")
    for name in CMP:
        tb = pd.DataFrame({f"{bn}≧{b:.0%}" if kind == "tan" else f"{bn}≧{b}":
                           [f"{G[(name, a, b)]['roi']:.0%}（{G[(name, a, b)]['n']}）" if G[(name, a, b)]["n"] >= 30 else "-" for a in A_]
                           for b in B_}, index=[f"{an}≧{a}" for a in A_])
        print(f"\n▼ {name}: 3年合計の回収率（点数）")
        print(tb.to_string())
    for cand in ["③全部", "③上位"]:
        cells = [(a, b) for a in A_ for b in B_ if min(G[("今のモデル", a, b)]["n"], G[(cand, a, b)]["n"]) >= 100]
        win = sum(G[(cand, a, b)]["roi"] > G[("今のモデル", a, b)]["roi"] for a, b in cells)
        print(f"\n★ {cand} が今のモデルを上回ったマス: {win} / {len(cells)}（両方とも100点以上のマス）")
    print(f"\n▼ 3年とも100%超・各年30点以上のマス")
    for name in CMP:
        ok = [f"{an}≧{a}・{bn}≧{b}: " + " / ".join(f"{y:.0%}（{n}）" for y, n in zip(G[(name, a, b)]["yr"], G[(name, a, b)]["ny"]))
              + f" → 3年 {G[(name, a, b)]['roi']:.0%}・的中率 {G[(name, a, b)]['hit']:.1%}"
              for a in A_ for b in B_ if min(G[(name, a, b)]["ny"]) >= 30 and all(y > 1 for y in G[(name, a, b)]["yr"])]
        print(f"  [{name}] {len(ok)}マス")
        for x in ok:
            print("    " + x)
    print(f"\n▼ 今の基準での比較（{'期待値1.2・1着率10%' if kind == 'tan' else '度合い1.6・ワイド確率0.2'}）")
    key = (1.2, 0.10) if kind == "tan" else (1.6, 0.20)
    for name in CMP:
        g = G[(name, *key)]
        print(f"  {name}: " + " / ".join(f"{Y}年 {y:.0%}（{n}）" for Y, y, n in zip(YEARS, g["yr"], g["ny"])) + f" → 3年 {g['roi']:.0%}・的中率 {g['hit']:.1%}")
print("\n※ 判断も払戻も確定オッズ。年の並びは " + " / ".join(map(str, YEARS)))